In [1]:
import pandas as pd
from src import database, request, taxonomy
import re
from bs4 import BeautifulSoup
import asyncio

In [2]:
df = pd.read_csv('data/cox2_th_cutadapt/cox2_th_cutadapt-meta.csv', sep='\t')
df

,accession,creation_date,title,authors,sequence_length,organism_name,tax_id_x,organism,organelle,mol_type,...,trimed_seq_len,tax_id_y,species,domain,kingdom,phylum,class,order,family,genus
0,MW006635.1,23-NOV-2020,Globisporangium irregulare isolate UM1087 cyto...,"Callaghan,S.E.",621,Globisporangium irregulare,36331,Globisporangium irregulare,mitochondrion,genomic DNA,...,315,36331,Globisporangium irregulare,Eukaryota,Stramenopiles,Oomycota,Peronosporomycetes,Pythiales,Pythiaceae,Globisporangium
1,PP444835.1,14-MAR-2024,Peronosclerospora maydis isolate Tabanan cytoc...,"Sugiarta,D., Susanta Wirya,G.A. and Agung Phab...",346,Peronosclerospora maydis,886949,Peronosclerospora maydis,mitochondrion,genomic DNA,...,308,886949,Peronosclerospora maydis,Eukaryota,Stramenopiles,Oomycota,Peronosporomycetes,Peronosporales,Peronosporaceae,Peronosclerospora
2,OP727296.1,11-NOV-2023,Peronosclerospora neglecta culture BRIP:71135 ...,"Muis,A., Ryley,M.J., Tan,Y.P., Suharjo,R., Non...",409,Peronosclerospora neglecta,3090892,Peronosclerospora neglecta,mitochondrion,genomic DNA,...,227,3090892,Peronosclerospora neglecta,Eukaryota,Stramenopiles,Oomycota,Peronosporomycetes,Peronosporales,Peronosporaceae,Peronosclerospora
3,KM819519.1,27-JAN-2015,Pythiogeton ramosum isolate SC2.7.2 cytochrome...,"Le,D.P., Smith,M.K. and Aitken,E.",614,Pythiogeton ramosum,1217586,Pythiogeton ramosum,mitochondrion,genomic DNA,...,307,1217586,Pythiogeton ramosum,Eukaryota,Stramenopiles,Oomycota,Peronosporomycetes,Pythiales,Pythiogetonaceae,Pythiogeton
4,DQ071412.1,22-JUL-2009,Phytophthora megasperma strain CH95 cytochrome...,"Villa,N.O., Kageyama,K., Asano,T. and Suga,H.",563,Phytophthora megasperma,4788,Phytophthora megasperma,mitochondrion,genomic DNA,...,277,4788,Phytophthora megasperma,Eukaryota,Stramenopiles,Oomycota,Peronosporomycetes,Peronosporales,Peronosporaceae,Phytophthora
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1292,MK602199.1,29-APR-2020,Globisporangium spinosum isolate C9 cytochrome...,"Ibanez,J.M., Obregon,V.G. and Favaro,M.A.",518,Globisporangium spinosum,2125998,Globisporangium spinosum,mitochondrion,genomic DNA,...,211,2125998,Globisporangium spinosum,Eukaryota,Stramenopiles,Oomycota,Peronosporomycetes,Pythiales,Pythiaceae,Globisporangium
1293,MK602203.1,29-APR-2020,Globisporangium irregulare isolate C13 cytochr...,"Ibanez,J.M., Obregon,V.G. and Favaro,M.A.",518,Globisporangium irregulare,36331,Globisporangium irregulare,mitochondrion,genomic DNA,...,211,36331,Globisporangium irregulare,Eukaryota,Stramenopiles,Oomycota,Peronosporomycetes,Pythiales,Pythiaceae,Globisporangium
1294,MK602204.1,29-APR-2020,Globisporangium irregulare isolate C14 cytochr...,"Ibanez,J.M., Obregon,V.G. and Favaro,M.A.",518,Globisporangium irregulare,36331,Globisporangium irregulare,mitochondrion,genomic DNA,...,211,36331,Globisporangium irregulare,Eukaryota,Stramenopiles,Oomycota,Peronosporomycetes,Pythiales,Pythiaceae,Globisporangium
1295,MK602213.1,29-APR-2020,Globisporangium irregulare isolate C24 cytochr...,"Ibanez,J.M., Obregon,V.G. and Favaro,M.A.",518,Globisporangium irregulare,36331,Globisporangium irregulare,mitochondrion,genomic DNA,...,211,36331,Globisporangium irregulare,Eukaryota,Stramenopiles,Oomycota,Peronosporomycetes,Pythiales,Pythiaceae,Globisporangium


In [ ]:
hosts = pd.Series(df['host'].dropna().unique())
hosts = hosts.str.replace(r'\(.*\)', '', regex=True)
hosts = hosts.str.replace('  ', ' ')
hosts = hosts.apply(database.sanitize_org_name)
def to_genus(name: str) -> str:
    parts = name.split(' ')[:2]
    # remove non-alphabetical characters
    parts = list(map(lambda x: re.sub(r'[^a-zA-Z]', '', x), parts))
    if len(parts) == 2 and (parts[1] == 'sp' or parts[1] == 'spp'):
        return parts[0]
    return ' '.join(parts)
hosts = hosts.apply(to_genus)
hosts = pd.Series(hosts.unique()).astype(str)
hosts

0      Solanum lycopersicum
1                  Zea mays
2                       dog
3           Brassica juncea
4                    citrus
               ...         
401       Knautia sylvatica
402        Anemone nemorosa
403       Fragaria ananassa
404       Citrullus lanatus
405                 Eustoma
Length: 406, dtype: object

In [4]:
client = request.HttpClient()

tax_ids = {}

for species in hosts:
    await asyncio.sleep(0.1)
    response = await client.esearch(species, 'taxonomy')
    id = None
    if response.result_count > 0:
        bs = BeautifulSoup(response.text_response, features="xml")
        id = int(bs.find('Id').string)
    tax_ids[species] = id

host_df = pd.DataFrame({'organism_name': hosts})
host_df

,organism_name
0,Solanum lycopersicum
1,Zea mays
2,dog
3,Brassica juncea
4,citrus
...,...
401,Knautia sylvatica
402,Anemone nemorosa
403,Fragaria ananassa
404,Citrullus lanatus


In [5]:
#host_df = pd.DataFrame(tax_ids)
tax_ids

{'Solanum lycopersicum': 4081,
 'Zea mays': 4577,
 'dog': 9615,
 'Brassica juncea': 3707,
 'citrus': 2706,
 'Barbula unguiculata': 96465,
 'Oryza sativa': 4530,
 'Anoectochilus roxburghii': 569774,
 'Chenopodium album': 3559,
 'Cancrinia discoidea': 99082,
 'Semiaquilegia adoxoides': 349366,
 'Rhododendron': 4346,
 'Malus': 3749,
 'Lepidium sativum': 33125,
 'Panax notoginseng': 44586,
 'cacao': 3641,
 'Eruca vesicaria': 180536,
 'Sisymbrium': 3729,
 'Isatis': 161755,
 'Cinnamomum aromaticum': 119260,
 'Sarga leiocladum': 91527,
 'Sorghum plumosum': 143049,
 'Spinacia oleracea': 3562,
 'Atriplex patula': 3551,
 'Atriplex littoralis': 543010,
 'Atriplex prostrata': 190330,
 'Beta vulgaris': 161934,
 'Spergula arvensis': 325515,
 'Spergularia rubra': 157641,
 'Polycarpon diphyllum': 431019,
 'Rumex acetosella': 61450,
 'Chenopodium bonushenricus': None,
 'Erucastrum nasturtiifolium': 308294,
 'Manihot': 3982,
 'Sinapis arvensis': 2982281,
 'Lepidium campestre': 65351,
 'Plantago aristata

In [ ]:
async def fetch_lineage(taxids):
    lineage_data = {}
    
    # Batch into groups of ~200 IDs
    batch_size = 50
    taxids = [str(id) for id in taxids if id is not None]
    for i in range(0, len(taxids), batch_size):
        batch = taxids[i:i+batch_size]
        response = await client._client.get(f'{client._base_url}efetch.fcgi', params={
            'db': 'taxonomy',
            "id": ",".join(batch),
            "retmode": "xml",
            "rettype": "text"
        })
        response.raise_for_status()
        with open(f'data/cox2_db_cutadapt-meta-{i}.xml', 'w') as f:
            f.write(response.text)
        await asyncio.sleep(0.1)

    return lineage_data
lineage_data = await fetch_lineage(list(tax_ids.values()))
lineage_data

{}

In [2]:
taxonomy.parse_taxonomy('data/cox2_db_cutadapt-meta-0.xml', 'data/cox2_db_cutadapt-meta-0.csv')

In [3]:
await taxonomy.from_organsim_names(df['host'], 'data/cox2_th_cutadapt/host-tax-{}.xml', 'data/cox2_th_cutadapt/host-tax.csv')